# Platelet Classification with Machine Learning (Python)

Predict patient survival (S vs NS) using single-cell platelet gene expression data.

**Models**: XGBoost, Random Forest, SVM, Logistic Regression, Gradient Boosting

In [ ]:
# Install packages if needed
# !pip install scanpy xgboost scikit-learn matplotlib seaborn

In [ ]:
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             roc_curve, auc, roc_auc_score, precision_recall_curve)
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded!")

## 1. Load Data

In [ ]:
# Load h5ad file
DATA_PATH = "/bigdata/godziklab/shared/Xinru/302005/302005_platelet_harmony_integrated.h5ad"
adata = sc.read_h5ad(DATA_PATH)

print(f"Loaded: {adata.n_obs} cells, {adata.n_vars} genes")
print(f"\nMetadata columns: {adata.obs.columns.tolist()}")

In [ ]:
# Check target variable
print("=== Death (Target) ===")
print(adata.obs['Death'].value_counts())

print("\n=== Category ===")
print(adata.obs['Category'].value_counts())

## 2. Data Preprocessing

In [ ]:
# Filter to S (Survivor) vs NS (Non-Survivor) only
adata_filtered = adata[adata.obs['Death'].isin(['S', 'NS'])].copy()
print(f"After filtering: {adata_filtered.n_obs} cells")
print(adata_filtered.obs['Death'].value_counts())

In [ ]:
# Prepare features (X) and labels (y)
# Use expression matrix
if hasattr(adata_filtered.X, 'toarray'):
    X = adata_filtered.X.toarray()  # Convert sparse to dense
else:
    X = adata_filtered.X

# Encode labels
le = LabelEncoder()
y = le.fit_transform(adata_filtered.obs['Death'])  # NS=0, S=1
print(f"Classes: {le.classes_}")
print(f"Feature matrix shape: {X.shape}")
print(f"Label distribution: NS={sum(y==0)}, S={sum(y==1)}")

In [ ]:
# Feature selection: top variable genes
N_TOP_GENES = 500

gene_vars = np.var(X, axis=0)
top_gene_idx = np.argsort(gene_vars)[-N_TOP_GENES:]
X_selected = X[:, top_gene_idx]
gene_names = adata_filtered.var_names[top_gene_idx].tolist()

print(f"Selected top {N_TOP_GENES} variable genes")
print(f"New feature shape: {X_selected.shape}")

In [ ]:
# Train/Test Split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

## 3. Train Multiple Models

In [ ]:
# Define models
models = {
    'XGBoost': XGBClassifier(
        n_estimators=100, max_depth=6, learning_rate=0.1,
        random_state=42, use_label_encoder=False, eval_metric='logloss'
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=10, random_state=42, n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42
    ),
    'Logistic Regression': LogisticRegression(
        max_iter=1000, random_state=42, n_jobs=-1
    ),
    'SVM': SVC(
        kernel='rbf', probability=True, random_state=42
    )
}

print(f"Models to train: {list(models.keys())}")

In [ ]:
# Train all models and collect results
results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    
    # Use scaled data for SVM and Logistic Regression
    if name in ['SVM', 'Logistic Regression']:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)
        y_prob = model.predict_proba(X_test_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]
    
    # Calculate metrics
    acc = accuracy_score(y_test, y_pred)
    auc_score = roc_auc_score(y_test, y_prob)
    
    results[name] = {
        'model': model,
        'y_pred': y_pred,
        'y_prob': y_prob,
        'accuracy': acc,
        'auc': auc_score
    }
    
    print(f"  Accuracy: {acc:.4f}, AUC: {auc_score:.4f}")

print("\n" + "="*50)
print("All models trained!")

## 4. Cross-Validation

In [ ]:
# 5-fold Cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}

print("Running 5-fold Cross-Validation...\n")

for name, model in models.items():
    if name in ['SVM', 'Logistic Regression']:
        scores = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='roc_auc')
    else:
        scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='roc_auc')
    
    cv_results[name] = scores
    print(f"{name}: AUC = {scores.mean():.4f} (+/- {scores.std()*2:.4f})")

## 5. Model Evaluation & Visualization

In [ ]:
# Summary table
summary_df = pd.DataFrame({
    'Model': list(results.keys()),
    'Accuracy': [results[m]['accuracy'] for m in results],
    'AUC': [results[m]['auc'] for m in results],
    'CV_AUC_Mean': [cv_results[m].mean() for m in results],
    'CV_AUC_Std': [cv_results[m].std() for m in results]
}).sort_values('AUC', ascending=False)

print("\n" + "="*60)
print("MODEL PERFORMANCE SUMMARY")
print("="*60)
print(summary_df.to_string(index=False))

In [ ]:
# Plot ROC Curves
plt.figure(figsize=(10, 8))

colors = ['#E41A1C', '#4DAF4A', '#377EB8', '#FF7F00', '#984EA3']

for (name, res), color in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    plt.plot(fpr, tpr, color=color, lw=2, 
             label=f"{name} (AUC = {res['auc']:.3f})")

plt.plot([0, 1], [0, 1], 'k--', lw=1)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves - Survival Prediction (S vs NS)', fontsize=14)
plt.legend(loc='lower right', fontsize=10)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('roc_curves_comparison.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved: roc_curves_comparison.png")

In [ ]:
# Plot Confusion Matrices
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, (name, res) in enumerate(results.items()):
    cm = confusion_matrix(y_test, res['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[idx],
                xticklabels=['NS', 'S'], yticklabels=['NS', 'S'])
    axes[idx].set_title(f"{name}\nAcc: {res['accuracy']:.3f}, AUC: {res['auc']:.3f}")
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('Actual')

# Hide empty subplot
axes[-1].axis('off')

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved: confusion_matrices.png")

In [ ]:
# CV Results Box Plot
cv_df = pd.DataFrame(cv_results)

plt.figure(figsize=(10, 6))
cv_df.boxplot()
plt.ylabel('AUC Score', fontsize=12)
plt.title('5-Fold Cross-Validation AUC Scores', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('cv_results_boxplot.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved: cv_results_boxplot.png")

## 6. Feature Importance

In [ ]:
# XGBoost Feature Importance
xgb_model = results['XGBoost']['model']
xgb_importance = pd.DataFrame({
    'Gene': gene_names,
    'Importance': xgb_model.feature_importances_
}).sort_values('Importance', ascending=False)

# Top 20 features
top_20 = xgb_importance.head(20)

plt.figure(figsize=(10, 8))
plt.barh(range(len(top_20)), top_20['Importance'].values, color='steelblue')
plt.yticks(range(len(top_20)), top_20['Gene'].values)
plt.xlabel('Importance Score', fontsize=12)
plt.title('XGBoost Feature Importance (Top 20 Genes)', fontsize=14)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('feature_importance_xgboost.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nTop 20 Important Genes (XGBoost):")
print(top_20.to_string(index=False))

In [ ]:
# Random Forest Feature Importance
rf_model = results['Random Forest']['model']
rf_importance = pd.DataFrame({
    'Gene': gene_names,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)

top_20_rf = rf_importance.head(20)

plt.figure(figsize=(10, 8))
plt.barh(range(len(top_20_rf)), top_20_rf['Importance'].values, color='forestgreen')
plt.yticks(range(len(top_20_rf)), top_20_rf['Gene'].values)
plt.xlabel('Importance Score', fontsize=12)
plt.title('Random Forest Feature Importance (Top 20 Genes)', fontsize=14)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('feature_importance_rf.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Consensus Top Genes (appear in both XGBoost and RF top 50)
top50_xgb = set(xgb_importance.head(50)['Gene'])
top50_rf = set(rf_importance.head(50)['Gene'])
consensus_genes = top50_xgb.intersection(top50_rf)

print(f"\nConsensus Top Genes (in both XGBoost & RF top 50): {len(consensus_genes)}")
print(sorted(consensus_genes))

## 7. Save Results

In [ ]:
# Save summary
summary_df.to_csv('model_comparison_summary.csv', index=False)
print("Saved: model_comparison_summary.csv")

# Save feature importance
xgb_importance.to_csv('feature_importance_xgboost.csv', index=False)
rf_importance.to_csv('feature_importance_rf.csv', index=False)
print("Saved: feature_importance_xgboost.csv")
print("Saved: feature_importance_rf.csv")

# Save consensus genes
pd.DataFrame({'Gene': sorted(consensus_genes)}).to_csv('consensus_top_genes.csv', index=False)
print("Saved: consensus_top_genes.csv")

## 8. Classification Report (Best Model)

In [ ]:
# Get best model
best_model_name = summary_df.iloc[0]['Model']
best_result = results[best_model_name]

print(f"\n{'='*60}")
print(f"BEST MODEL: {best_model_name}")
print(f"{'='*60}")
print(f"\nAccuracy: {best_result['accuracy']:.4f}")
print(f"AUC: {best_result['auc']:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, best_result['y_pred'], 
                            target_names=['NS (Non-Survivor)', 'S (Survivor)']))

## Summary

This notebook demonstrated:
1. Loading h5ad single-cell data
2. Preprocessing for ML (feature selection, scaling)
3. Training 5 classifiers (XGBoost, RF, GB, LR, SVM)
4. 5-fold cross-validation
5. ROC curves and confusion matrices
6. Feature importance analysis

**Output files:**
- `roc_curves_comparison.png`
- `confusion_matrices.png`
- `cv_results_boxplot.png`
- `feature_importance_xgboost.png`
- `feature_importance_rf.png`
- `model_comparison_summary.csv`
- `feature_importance_*.csv`
- `consensus_top_genes.csv`